In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import datasets # tensorflow가 제공하는 데이터셋을 사용하기 위해 import 한다.
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
# tensorflow에서 합성곱 신경망(Convolution Neural Nerwork, CNN)을 사용하기 위해 추가한다.
from tensorflow.keras.layers import Conv2D # 신경망에 추가할 합성곱 레이어를 생성하기 위해 import 한다.
from tensorflow.keras.layers import MaxPool2D # 신경망에 추가할 맥스 풀링 레이어를 생성하기 위해 import 한다.
from tensorflow.keras.layers import Flatten # 신경망의 최종 출력층에 벡터 형태(1차원)를 만들기 위해 import 한다.
from tensorflow.keras.layers import Dropout # 신경망에 추가할 드롭다운 레이어를 생성하기 위해 import 한다.

합성곱 신경망(Convolution Neural Nerwork, CNN)

합성곱 신경망은 흔히 CNN이라고 부르는 합성곱이라는 연산을 사용하는 신경망이다. CNN은 실제 여러 분야에서 사용되는 방법으로 특히 이미지 분류 작업에서 좋은 성능ㅇ를 보여준다.

합성곱 연산은 아래와 같은 연산을 의미한다.

$$y(i) = (x \times w)(i) = \sum_{k=1}^{\infty} x(i - k)w(k)$$

패딩(padding)

패딩은 입력 데이터 주변을 특정 값으로 채우는 것을 말한다.

<img src="./cnn_1.png" width="600" align="left" />

신경망에 커널을 적용하면 층이 깊어질수록 데이터의 차원은 점점 줄어든다. 5 * 5 차원의 입력 데이터에 2 * 2 차원의 커널을 합성곱했을 때 출력 데이터는 4 * 4 차원으로 입력 데이터보다 출력 데이터의 차원이 줄어든다. 이렇듯, 입력 데이터에 커널을 합성곱한 후 출력 데이터의 차원이 줄어드는 현상을 방지하기 위해서 패딩이라는 방법을 사용한다.

스트라이드(stride)

스트라이드는 한 번 합성곱 연산을 한 후 다음 계산 영역을 선택할 때 얼마나 이동할지 간격을 정하는 것이다.

<img src="./cnn_2.png" width="400" align="left" />

풀링(polling)

풀링은 데이터의 차원을 줄이는 방법이다.

<img src="./cnn_3.png" width="250" align="left" />

맥스 풀링이란 해당 영역에서 가장 큰 값을 선택하는 방법이다.

합성곱 신경망을 이용해서 손글씨 인식 모델을 생성해 본다.

합성곱 신경망에 사용할 데이터 준비

In [9]:
# mnist 손글씨 데이터는 학습 데이터와 테스트 데이터가 튜플로 구분되서 저장되어 있다.
# datasets.mnist.load_data() 메소드는 (학습 피쳐 데이터, 학습 레이블 데이터)와 (테스트 피쳐 데이터, 테스트 레이블 데이터)를 튜플 형태로 묶어서 리턴한다.
# 손글씨 숫자 데이터셋을 학습 데이터와 테스트 데이터로 저장한다.
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# 원본 데이터 확인
# 학습 피쳐 데이터의 차원을 확인하면 (이미지 개수, 이미지 1건의 행의 개수, 이미지 1건의 열의 개수) 형태로 나온다. 즉, 학습 피쳐 데이터는 28행 * 28열의 이미지
# 60,000개로 구성된 배열이라는 것을 알 수 있다.
print(type(x_train), x_train.shape)
print(type(y_train), y_train.shape) # 학습 레이블 데이터는 스칼라값 60,000개로 이루어진 벡터이다.
print(type(x_test), x_test.shape) # 테스트 피쳐 데이터는 28행 * 28열 이미지 10,000개로 구성된 배열이라는 것을 알 수 있다.
print(type(y_test), y_test.shape) # 테스트 레이블 데이터는 스칼라값 10,000개로 이루어진 벡터이다.

<class 'numpy.ndarray'> (60000, 28, 28)
<class 'numpy.ndarray'> (60000,)
<class 'numpy.ndarray'> (10000, 28, 28)
<class 'numpy.ndarray'> (10000,)


원본 데이터 시각화

In [10]:
# 레이블 종류 확인 => 회귀 문제인지 분류 문제인지 파악할 수 있고 분류 문제라면 몇 개의 클래스로 구분되는지 알 수 있다.

In [11]:
# 피쳐 데이터 차원 변경

In [12]:
# 피쳐 데이터 스케일 조정

In [13]:
# 레이블 데이터 원-핫 인코딩

합성곱 신경망 모델을 만든다.